In [12]:
import sys
import os
sys.path.append(os.path.abspath("../../src/vbpca_py"))
from errpca_pt import errpca_pt 

In [14]:
# Test 1: Basic Functionality Test
print('Test 1: Basic Functionality Test')

import numpy as np
from scipy.sparse import csr_matrix

# Fixed data matrices
X = csr_matrix(np.array([[1, 0, 0],
                         [0, 2, 0],
                         [0, 0, 3]]))

A = np.array([[1], [2], [3]])   # Shape (3 x 1)
S = np.array([[4, 5, 6]])       # Shape (1 x 3)

# Prepare CSR components
X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

# Compute error using Python function
result = errpca_pt(X_data, X_indices, X_indptr, A, S)

# Reconstruct sparse matrix from result
errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])

# Display result
print('errMx_python:')
print(errMx_python.toarray())


Test 1: Basic Functionality Test
errMx_python:
[[ -3.   0.   0.]
 [  0.  -8.   0.]
 [  0.   0. -15.]]


In [15]:
# Test 2: Zero Matrix Input
print('Test 2: Zero Matrix Input')

import numpy as np
from scipy.sparse import csr_matrix

# Create zero sparse matrix X
X = csr_matrix((3, 3))

# Fixed matrices A and S
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])  # Shape (3 x 2)

S = np.array([[7, 8, 9],
              [10, 11, 12]])  # Shape (2 x 3)

# Prepare CSR components
X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

# Compute error using Python function
# No change needed in the function call since shape is now correctly determined
result = errpca_pt(X_data, X_indices, X_indptr, A, S)

# Reconstruct sparse matrix from result
errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])

# Expected result is zero sparse matrix
errMx_expected = csr_matrix((3, 3))

# Verify the result
difference = np.linalg.norm(errMx_python.toarray() - errMx_expected.toarray(), ord='fro')
print(f'Difference: {difference:e}')

if difference < 1e-10:
    print('Test 2 passed: Function handles zero matrix correctly.')
else:
    print('Test 2 failed: Function does not handle zero matrix as expected.')

print(errMx_python.toarray())


Test 2: Zero Matrix Input
Difference: 0.000000e+00
Test 2 passed: Function handles zero matrix correctly.
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [16]:
# Test 3: Single Non-Zero Element
print('Test 3: Single Non-Zero Element')

import numpy as np
from scipy.sparse import csr_matrix

# Create sparse matrix with one non-zero element
X = csr_matrix((3, 3))
X[1, 1] = 5  # Zero-based indexing in Python

# Fixed matrices A and S
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])

S = np.array([[7, 8, 9],
              [10, 11, 12]])

# Prepare CSR components
X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

# Compute error using Python function
result = errpca_pt(X_data, X_indices, X_indptr, A, S)

# Reconstruct sparse matrix from result
errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])

# Compute expected error

# Compute A * S (AS)
AS = A @ S  # Shape (3 x 3)

# Get the non-zero indices of X
rows, cols = X.nonzero()

# Compute the error values at the non-zero positions of X
err_values = X.data - AS[rows, cols]

# Create the expected error sparse matrix
errMx_expected = csr_matrix((err_values, (rows, cols)), shape=X.shape)

# Verify the result
difference = np.linalg.norm(errMx_python.toarray() - errMx_expected.toarray(), ord='fro')
print(f'Difference: {difference:e}')

if difference < 1e-10:
    print('Test 3 passed: Function handles single non-zero element correctly.')
else:
    print('Test 3 failed: Function does not handle single non-zero element as expected.')
print(errMx_python.toarray())


Test 3: Single Non-Zero Element
Difference: 0.000000e+00
Test 3 passed: Function handles single non-zero element correctly.
[[  0.   0.   0.]
 [  0. -63.   0.]
 [  0.   0.   0.]]


In [17]:
# Test 6A: Very Low Density
print('Test 6A: Very Low Density')

import numpy as np
from scipy.sparse import random as sparse_random
from scipy.sparse import csr_matrix

n1, n2, ncomp = 100, 100, 10
density = 0.0001  # Very low density

X = sparse_random(n1, n2, density=density, format='csr', dtype=np.float64, data_rvs=np.random.randn)

A = np.random.randn(n1, ncomp)
S = np.random.randn(ncomp, n2)

X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

result = errpca_pt(X_data, X_indices, X_indptr, A, S)

errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])

# Compute expected error
AS = A @ S
rows, cols = X.nonzero()
err_values = X.data - AS[rows, cols]
errMx_expected = csr_matrix((err_values, (rows, cols)), shape=X.shape)

difference = np.linalg.norm(errMx_python.toarray() - errMx_expected.toarray(), ord='fro')
print(f'Difference: {difference:e}')

if difference < 1e-8:
    print('Test 6A passed: Function handles very low density matrices correctly.')
else:
    print('Test 6A failed: Function does not handle very low density matrices as expected.')
print(errMx_python.toarray())


Test 6A: Very Low Density
Difference: 2.220446e-16
Test 6A passed: Function handles very low density matrices correctly.
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [18]:
# Test 5: Non-Square Matrix
print('Test 5: Non-Square Matrix')

import numpy as np
from scipy.sparse import csr_matrix

# Create a non-square sparse matrix X (5 rows x 4 columns)
X = csr_matrix(np.array([
    [1, 0, 0, 0],
    [0, 2, 0, 0],
    [0, 0, 3, 0],
    [0, 0, 0, 4],
    [0, 0, 0, 0]
]))

# Fixed matrices A and S
A = np.array([[1,  2],
              [3,  4],
              [5,  6],
              [7,  8],
              [9, 10]])  # Shape (5 x 2)

S = np.array([[11, 12, 13, 14],
              [15, 16, 17, 18]])  # Shape (2 x 4)

# Prepare CSR components
X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

# Compute error using Python function
result = errpca_pt(X_data, X_indices, X_indptr, A, S)

# Reconstruct sparse matrix from result
errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])

# Compute expected error

# Compute A * S (AS)
AS = A @ S  # Shape (5 x 4)

# Get the non-zero indices of X
rows, cols = X.nonzero()

# Compute the error values at the non-zero positions
err_values = X.data - AS[rows, cols]

# Create the expected error sparse matrix
errMx_expected = csr_matrix((err_values, (rows, cols)), shape=X.shape)

# Verify the result
difference = np.linalg.norm(errMx_python.toarray() - errMx_expected.toarray(), ord='fro')
print(f'Difference: {difference:e}')

if difference < 1e-10:
    print('Test 5 passed: Function handles non-square matrices correctly.')
else:
    print('Test 5 failed: Function does not handle non-square matrices as expected.')
print(errMx_python.toarray())


Test 5: Non-Square Matrix
Difference: 0.000000e+00
Test 5 passed: Function handles non-square matrices correctly.
[[ -40.    0.    0.    0.]
 [   0.  -98.    0.    0.]
 [   0.    0. -164.    0.]
 [   0.    0.    0. -238.]
 [   0.    0.    0.    0.]]


In [19]:
# Test 6B: Very High Density
print('Test 6B: Very High Density')

import numpy as np
from scipy.sparse import random as sparse_random
from scipy.sparse import csr_matrix

n1, n2, ncomp = 100, 100, 10
density = 0.99  # Very high density

X = sparse_random(n1, n2, density=density, format='csr', dtype=np.float64, data_rvs=np.random.randn)

A = np.random.randn(n1, ncomp)
S = np.random.randn(ncomp, n2)

X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

result = errpca_pt(X_data, X_indices, X_indptr, A, S)

errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])

# Compute expected error
AS = A @ S
rows, cols = X.nonzero()
err_values = X.data - AS[rows, cols]
errMx_expected = csr_matrix((err_values, (rows, cols)), shape=X.shape)

difference = np.linalg.norm(errMx_python.toarray() - errMx_expected.toarray(), ord='fro')
print(f'Difference: {difference:e}')

if difference < 1e-8:
    print('Test 6B passed: Function handles very high density matrices correctly.')
else:
    print('Test 6B failed: Function does not handle very high density matrices as expected.')
print(errMx_python.toarray())


Test 6B: Very High Density
Difference: 4.094356e-14
Test 6B passed: Function handles very high density matrices correctly.
[[-4.42096395  3.76228348  1.62233485 ... -2.3570467   2.51498478
  -2.74822429]
 [-1.41159052  1.78130123 -4.99487309 ...  2.13547044  0.
   0.97176843]
 [-0.28153456 -1.08577079  1.7905882  ...  3.78892289 -2.26126895
  -3.55162728]
 ...
 [-6.40656036  0.08090871 -0.80086317 ...  4.08895017 -2.17285593
  -0.45661461]
 [ 1.36841155 -4.45475188 -1.24699565 ...  2.3422497  -1.96422639
  -1.55648553]
 [ 0.05560975  0.0608686  -0.6183164  ...  2.53358862 -4.85576548
  -1.89682478]]


In [21]:
# Test 8: Multi-threading Test
print('Test 8: Multi-threading Test')

import numpy as np
from scipy.sparse import random as sparse_random
from scipy.sparse import csr_matrix
import time

n1, n2, ncomp = 1000, 1000, 10
density = 0.01

X = sparse_random(n1, n2, density=density, format='csr', dtype=np.float64, data_rvs=np.random.randn)

A = np.random.randn(n1, ncomp)
S = np.random.randn(ncomp, n2)

X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

numCPU = 4  # Number of threads

# Compute error using Python function
start_time = time.time()
result = errpca_pt(X_data, X_indices, X_indptr, A, S, numCPU)
multi_thread_time = time.time() - start_time

errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])

# For comparison, run single-threaded version
start_time = time.time()
result_single = errpca_pt(X_data, X_indices, X_indptr, A, S, numCPU=1)
single_thread_time = time.time() - start_time

errMx_python_single = csr_matrix((result_single['data'], result_single['indices'], result_single['indptr']), shape=result['shape'])

# Verify correctness
difference = np.linalg.norm(errMx_python.toarray() - errMx_python_single.toarray(), ord='fro')
print(f'Difference between multi-threaded and single-threaded results: {difference:e}')

print(f'Multi-threaded execution time: {multi_thread_time:.2f} seconds')
print(f'Single-threaded execution time: {single_thread_time:.2f} seconds')

if difference < 1e-8:
    print('Test 8 passed: Function handles multi-threading correctly.')
else:
    print('Test 8 failed: Function does not handle multi-threading as expected.')
print(errMx_python.toarray())


Test 8: Multi-threading Test
Difference between multi-threaded and single-threaded results: 0.000000e+00
Multi-threaded execution time: 0.00 seconds
Single-threaded execution time: 0.00 seconds
Test 8 passed: Function handles multi-threading correctly.
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [22]:
# Test 9: Empty Input Matrix
print('Test 9: Empty Input Matrix')

import numpy as np
from scipy.sparse import csr_matrix

# Create empty sparse matrix
X = csr_matrix((0, 0))

A = np.array([]).reshape(0, 0)
S = np.array([]).reshape(0, 0)

X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

try:
    result = errpca_pt(X_data, X_indices, X_indptr, A, S)
    errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])
    if errMx_python.nnz == 0:
        print('Test 9 passed: Function handles empty input matrix correctly.')
    else:
        print('Test 9 failed: Function does not handle empty input matrix as expected.')
except Exception as e:
    print('Test 9 failed: Function raised an error with empty input matrix.')
    print(f'Error message: {e}')
print(errMx_python.toarray())


Test 9: Empty Input Matrix
Test 9 passed: Function handles empty input matrix correctly.
[]


In [23]:
# Test 10: Input Matrices with NaN or Inf Values
print('Test 10: Input Matrices with NaN or Inf Values')

import numpy as np
from scipy.sparse import csr_matrix

n1, n2, ncomp = 5, 5, 3
density = 0.5

X = csr_matrix((n1, n2))
X_data = np.random.randn(int(n1 * n2 * density))
X_indices = np.random.choice(n2, size=X_data.size)
X_indptr = np.zeros(n1 + 1, dtype=int)
X_indptr[1:] = np.cumsum(np.random.poisson(lam=1, size=n1))
X_indptr[-1] = X_data.size
X = csr_matrix((X_data, X_indices, X_indptr), shape=(n1, n2))

# Introduce NaN and Inf values
X.data[0] = np.nan
X.data[1] = np.inf

A = np.random.randn(n1, ncomp)
S = np.random.randn(ncomp, n2)

X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

try:
    result = errpca_pt(X_data, X_indices, X_indptr, A, S)
    errMx_python = csr_matrix((result['data'], result['indices'], result['indptr']), shape=result['shape'])
    print('Test 10 passed: Function handles NaN and Inf values.')
except Exception as e:
    print('Test 10 failed: Function raised an error with NaN or Inf values.')
    print(f'Error message: {e}')
print(errMx_python.toarray())


Test 10: Input Matrices with NaN or Inf Values
Test 10 passed: Function handles NaN and Inf values.
[[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [        nan  0.          0.          0.          0.        ]
 [-0.24398053         inf  3.68852188  1.76213601  1.09007001]]


In [24]:
# Simplified Test 7: Negative Values
print('Simplified Test 7: Negative Values')

import numpy as np
from scipy.sparse import csr_matrix

# Define small matrices with negative values
n1, n2, ncomp = 2, 2, 1

# Create sparse matrix X with negative values
X_data = np.array([-1, -2], dtype=np.float64)
X_indices = np.array([0, 1], dtype=np.int32)  # Column indices
X_indptr = np.array([0, 1, 2], dtype=np.int32)  # Row pointers
X_shape = (n1, n2)
X = csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)

# Define matrices A and S with negative values
A = np.array([[-0.5],
              [-1.5]])  # Shape (2 x 1)

S = np.array([[-2, -3]])  # Shape (1 x 2)

# Prepare CSR components of X for the function
X_data = X.data
X_indices = X.indices
X_indptr = X.indptr

# Compute the error using the Python function
errpca_result = errpca_pt(X_data, X_indices, X_indptr, A, S)

# Reconstruct the error matrix from the result
errMx_python = csr_matrix((errpca_result['data'], errpca_result['indices'], errpca_result['indptr']), shape=errpca_result['shape'])

# Compute expected error
AS = A @ S  # Matrix multiplication
rows, cols = X.nonzero()
err_values_expected = X.data - AS[rows, cols]
errMx_expected = csr_matrix((err_values_expected, (rows, cols)), shape=X.shape)

# Verify the result
difference = np.linalg.norm(errMx_python.toarray() - errMx_expected.toarray(), ord='fro')
print(f'Difference: {difference:e}')

if difference < 1e-10:
    print('Simplified Test 7 passed: Function handles negative values correctly.')
else:
    print('Simplified Test 7 failed: Function does not handle negative values as expected.')

# Optionally, display the error matrix
print('errMx_python as an array:')
print(errMx_python.toarray())


Simplified Test 7: Negative Values
Difference: 0.000000e+00
Simplified Test 7 passed: Function handles negative values correctly.
errMx_python as an array:
[[-2.   0. ]
 [ 0.  -6.5]]
